In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

C:\Users\aengland\Anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\aengland\Anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-12-17 09:43:23.618897


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20241112-simple-model-test
Task: 01_get_targets
Subtask: 02_regression


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query_normal.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('SELECT\n'
 '\tbigAccountId,\n'
 '\tdtmBooking as dtmFunded,\n'
 '\tfltNetChgOff,\n'
 '\tMonthEndDate\n'
 'FROM riskdb.accountingReports.tblAccounting_LoanCOandNA_ME')


### Write into df

In [7]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# make col
df['bktype'] = 'nobk'

# show
df

<timed exec>:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


Wall time: 46.4 s


,bigAccountId,dtmFunded,fltNetChgOff,MonthEndDate,bktype
0,370217,2006-08-30 17:12:22,6796.08,2019-12-31,nobk
1,306072,2005-10-11 17:27:40,17702.55,2019-12-31,nobk
2,245270,2004-10-13 00:00:00,4755.88,2019-12-31,nobk
3,196070,2002-07-29 00:00:00,9161.37,2019-12-31,nobk
4,239071,2004-08-26 00:00:00,15970.30,2019-12-31,nobk
...,...,...,...,...,...
6123792,1864006,2015-01-14 15:04:58,5049.84,2024-11-30,nobk
6123793,1852393,2014-12-04 11:16:16,982.08,2024-11-30,nobk
6123794,1855867,2014-12-11 13:43:41,5030.75,2024-11-30,nobk
6123795,1865860,2014-12-30 13:24:25,11085.89,2024-11-30,nobk


### Read query

In [8]:
str_filepath = './sql/query_bk.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('SELECT\n'
 '\tbigAccountId,\n'
 '\tdtmFunded,\n'
 '\tmnyNetGainLoss as fltNetChgOff,\n'
 '\tdtmRunDate as MonthEndDate\n'
 'FROM riskdb.accountingReports.tblAccounting_ReportV11_ME')


### Write into df

In [9]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df_tmp = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# make col
df_tmp['bktype'] = 'bk'

# show
df_tmp

<timed exec>:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


Wall time: 741 ms


,bigAccountId,dtmFunded,fltNetChgOff,MonthEndDate,bktype
0,1427393,2013-12-30 17:26:10.000,0.00,2019-12-31,bk
1,998239,2012-09-28 16:20:09.000,16.00,2019-12-31,bk
2,1234987,2013-08-07 15:01:13.000,22.69,2019-12-31,bk
3,854538,2012-01-27 14:26:10.000,425.00,2019-12-31,bk
4,1670841,2014-07-22 15:44:21.000,112.27,2019-12-31,bk
...,...,...,...,...,...
137593,5765303,2021-11-12 13:19:13.927,778.48,2024-11-30,bk
137594,5774350,2021-10-08 14:56:53.467,657.05,2024-11-30,bk
137595,5790780,2021-11-09 11:03:59.943,0.00,2024-11-30,bk
137596,5794172,2021-10-28 13:15:49.387,14791.42,2024-11-30,bk


### Concatenate

In [10]:
df = pd.concat([df, df_tmp])
del df_tmp
# show
df

,bigAccountId,dtmFunded,fltNetChgOff,MonthEndDate,bktype
0,370217,2006-08-30 17:12:22.000,6796.08,2019-12-31,nobk
1,306072,2005-10-11 17:27:40.000,17702.55,2019-12-31,nobk
2,245270,2004-10-13 00:00:00.000,4755.88,2019-12-31,nobk
3,196070,2002-07-29 00:00:00.000,9161.37,2019-12-31,nobk
4,239071,2004-08-26 00:00:00.000,15970.30,2019-12-31,nobk
...,...,...,...,...,...
137593,5765303,2021-11-12 13:19:13.927,778.48,2024-11-30 00:00:00,bk
137594,5774350,2021-10-08 14:56:53.467,657.05,2024-11-30 00:00:00,bk
137595,5790780,2021-11-09 11:03:59.943,0.00,2024-11-30 00:00:00,bk
137596,5794172,2021-10-28 13:15:49.387,14791.42,2024-11-30 00:00:00,bk


### Convert funding month and month end date to first of each month

In [11]:
df['dtmFunded_first'] = df['dtmFunded'].dt.to_period('M').dt.to_timestamp()
df['MonthEndDate'] = pd.to_datetime(df['MonthEndDate'])
df['MonthEndDate_first'] = df['MonthEndDate'].dt.to_period('M').dt.to_timestamp()
df

,bigAccountId,dtmFunded,fltNetChgOff,MonthEndDate,bktype,dtmFunded_first,MonthEndDate_first
0,370217,2006-08-30 17:12:22.000,6796.08,2019-12-31,nobk,2006-08-01,2019-12-01
1,306072,2005-10-11 17:27:40.000,17702.55,2019-12-31,nobk,2005-10-01,2019-12-01
2,245270,2004-10-13 00:00:00.000,4755.88,2019-12-31,nobk,2004-10-01,2019-12-01
3,196070,2002-07-29 00:00:00.000,9161.37,2019-12-31,nobk,2002-07-01,2019-12-01
4,239071,2004-08-26 00:00:00.000,15970.30,2019-12-31,nobk,2004-08-01,2019-12-01
...,...,...,...,...,...,...,...
137593,5765303,2021-11-12 13:19:13.927,778.48,2024-11-30,bk,2021-11-01,2024-11-01
137594,5774350,2021-10-08 14:56:53.467,657.05,2024-11-30,bk,2021-10-01,2024-11-01
137595,5790780,2021-11-09 11:03:59.943,0.00,2024-11-30,bk,2021-11-01,2024-11-01
137596,5794172,2021-10-28 13:15:49.387,14791.42,2024-11-30,bk,2021-10-01,2024-11-01


### Get months on books

In [12]:
df['years'] = df['MonthEndDate_first'].dt.year - df['dtmFunded_first'].dt.year
# convert to months
df['months'] = df['years'] * 12
# get difference in months
df['months_tmp'] = df['MonthEndDate_first'].dt.month - df['dtmFunded_first'].dt.month
# get mob
df['months_on_books'] = df['months'] + df['months_tmp']
# show
df

,bigAccountId,dtmFunded,fltNetChgOff,MonthEndDate,bktype,dtmFunded_first,MonthEndDate_first,years,months,months_tmp,months_on_books
0,370217,2006-08-30 17:12:22.000,6796.08,2019-12-31,nobk,2006-08-01,2019-12-01,13,156,4,160
1,306072,2005-10-11 17:27:40.000,17702.55,2019-12-31,nobk,2005-10-01,2019-12-01,14,168,2,170
2,245270,2004-10-13 00:00:00.000,4755.88,2019-12-31,nobk,2004-10-01,2019-12-01,15,180,2,182
3,196070,2002-07-29 00:00:00.000,9161.37,2019-12-31,nobk,2002-07-01,2019-12-01,17,204,5,209
4,239071,2004-08-26 00:00:00.000,15970.30,2019-12-31,nobk,2004-08-01,2019-12-01,15,180,4,184
...,...,...,...,...,...,...,...,...,...,...,...
137593,5765303,2021-11-12 13:19:13.927,778.48,2024-11-30,bk,2021-11-01,2024-11-01,3,36,0,36
137594,5774350,2021-10-08 14:56:53.467,657.05,2024-11-30,bk,2021-10-01,2024-11-01,3,36,1,37
137595,5790780,2021-11-09 11:03:59.943,0.00,2024-11-30,bk,2021-11-01,2024-11-01,3,36,0,36
137596,5794172,2021-10-28 13:15:49.387,14791.42,2024-11-30,bk,2021-10-01,2024-11-01,3,36,1,37


### Save as parquet

In [13]:
%%time

# save
str_filename = 'df_loss.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

Wall time: 30.3 s


### Upload to s3

In [14]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 24.4 s


### Clean-up

In [15]:
os.remove(str_local_path)